# Análisis del Modelo de Datos y Punto de Equilibrio — workspace.innovalab

Este notebook analiza el modelo de datos de `workspace.innovalab`, calcula el punto de equilibrio para cada producto y grafica los resultados. Los datos ficticios se cargan desde el notebook [Ingesta Datos Innovalab](#notebook-3593257601905723) y los datos del documento "Analisis basico" desde el SQL query `data_ingest_Pan_artesano`.

## Modelo de Datos

### Tablas y sus Relaciones

| Tabla | PK | FK → | Descripción |
|---|---|---|---|
| `profiles` | `id` | — | Usuarios del sistema |
| `businesses` | `id` | `owner_user_id` → `profiles.id` | Negocios PyME |
| `products` | `id` | `business_id` → `businesses.id` | Productos de cada negocio |
| `business_cost_lines` | `id` | `business_id` → `businesses.id` | Líneas de costo a nivel negocio (fijos y variables) |
| `calculations` | `id` | `user_id` → `profiles.id`, `business_id` → `businesses.id`, `product_id` → `products.id` | Cálculos de costeo por producto |
| `costing_setup` | `calculation_id` | `calculation_id` → `calculations.id` | Configuración del cálculo (moneda, período, volumen) |
| `scenarios` | `id` | `calculation_id` → `calculations.id` | Escenarios (base, optimista, pesimista) |
| `calc_cost_lines` | `id` | `calculation_id` → `calculations.id`, `scenario_id` → `scenarios.id`, `source_business_cost_id` → `business_cost_lines.id` | Líneas de costo específicas del cálculo |
| `pricing_inputs` | `calculation_id` | `calculation_id` → `calculations.id` | Configuración de precios (margen o precio manual) |
| `computed_results` | `id` | `calculation_id` → `calculations.id`, `scenario_id` → `scenarios.id` | Resultados del punto de equilibrio |

### Orden de Inserción (padres → hijos)

1. **`profiles`** — sin dependencias
2. **`businesses`** — depende de `profiles`
3. **`products`** — depende de `businesses`
4. **`business_cost_lines`** — depende de `businesses`
5. **`calculations`** — depende de `profiles`, `businesses`, `products`
6. **`costing_setup`** — depende de `calculations`
7. **`scenarios`** — depende de `calculations`
8. **`calc_cost_lines`** — depende de `calculations`, `scenarios`, `business_cost_lines`
9. **`pricing_inputs`** — depende de `calculations`
10. **`computed_results`** — depende de `calculations`, `scenarios`

### Lógica del Punto de Equilibrio

- **`total_fixed`** = SUMA de `calc_cost_lines.amount` WHERE `behavior = 'fixed'`
- **`variable_unit`** = SUMA de `calc_cost_lines.amount` WHERE `behavior = 'variable'`
- **`unit_cost`** = `total_fixed / estimated_volume` + `variable_unit`
- **`price`** = `manual_price` (si no es NULL) ó `unit_cost × (1 + expected_margin_pct)`
- **`contribution_margin_unit`** = `price - variable_unit`
- **`break_even_units`** = `total_fixed / contribution_margin_unit`
- **`break_even_sales`** = `break_even_units × price`
- **`total_cost`** = `total_fixed + (variable_unit × estimated_volume)`

In [0]:
# ============================================================
# COMPUTED_RESULTS: calcular punto de equilibrio
# ============================================================

# total_fixed  = SUM(amount) WHERE behavior = 'fixed'
# variable_unit = SUM(amount) WHERE behavior = 'variable'
# unit_cost    = total_fixed / estimated_volume + variable_unit
# price        = manual_price COALESCE unit_cost * (1 + expected_margin_pct)
# contribution_margin_unit = price - variable_unit
# break_even_units = total_fixed / contribution_margin_unit
# break_even_sales = break_even_units * price
# total_cost   = total_fixed + variable_unit * estimated_volume

spark.sql("""
INSERT INTO workspace.innovalab.computed_results
SELECT
    uuid() AS id,
    t.calculation_id,
    t.scenario_id,
    CAST(t.total_fixed AS DECIMAL(18,4)),
    CAST(t.variable_unit AS DECIMAL(18,4)),
    CAST(t.total_fixed + t.variable_unit * t.estimated_volume AS DECIMAL(18,4)) AS total_cost,
    CAST(t.total_fixed / t.estimated_volume + t.variable_unit AS DECIMAL(18,4)) AS unit_cost,
    CAST(t.price - t.variable_unit AS DECIMAL(18,4)) AS contribution_margin_unit,
    CASE WHEN t.price - t.variable_unit > 0
         THEN CAST(t.total_fixed / (t.price - t.variable_unit) AS DECIMAL(18,4))
         ELSE CAST(0 AS DECIMAL(18,4)) END AS break_even_units,
    CASE WHEN t.price - t.variable_unit > 0
         THEN CAST(t.total_fixed / (t.price - t.variable_unit) * t.price AS DECIMAL(18,4))
         ELSE CAST(0 AS DECIMAL(18,4)) END AS break_even_sales,
    current_timestamp() AS computed_at,
    NULL AS results_payload
FROM (
    SELECT
        c.id AS calculation_id,
        sc.id AS scenario_id,
        tf.total_fixed,
        vu.variable_unit,
        cs.estimated_volume,
        COALESCE(
            pi.manual_price,
            (tf.total_fixed / cs.estimated_volume + vu.variable_unit) * (1 + pi.expected_margin_pct)
        ) AS price
    FROM workspace.innovalab.calculations c
    JOIN workspace.innovalab.scenarios sc ON sc.calculation_id = c.id AND sc.is_base = true
    JOIN workspace.innovalab.costing_setup cs ON cs.calculation_id = c.id
    JOIN workspace.innovalab.pricing_inputs pi ON pi.calculation_id = c.id
    JOIN (
        SELECT calculation_id, scenario_id, SUM(amount) AS total_fixed
        FROM workspace.innovalab.calc_cost_lines
        WHERE behavior = 'fixed'
        GROUP BY calculation_id, scenario_id
    ) tf ON tf.calculation_id = c.id AND tf.scenario_id = sc.id
    JOIN (
        SELECT calculation_id, scenario_id, SUM(amount) AS variable_unit
        FROM workspace.innovalab.calc_cost_lines
        WHERE behavior = 'variable'
        GROUP BY calculation_id, scenario_id
    ) vu ON vu.calculation_id = c.id AND vu.scenario_id = sc.id
) t
""")
count = spark.sql("SELECT COUNT(*) AS n FROM workspace.innovalab.computed_results").collect()[0]["n"]
print(f"✓ {count} computed_results calculados")

In [0]:
# ============================================================
# RESUMEN DE DATOS INSERTADOS
# ============================================================

tables = ["profiles","businesses","products","business_cost_lines","calculations",
          "costing_setup","scenarios","calc_cost_lines","pricing_inputs","computed_results"]
print("── Conteo de registros por tabla ──")
for t in tables:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM workspace.innovalab.{t}").collect()[0]["n"]
    print(f"  {t}: {n}")

print("\n── Muestra de computed_results (primeros 20) ──")
display(spark.sql("""
    SELECT c.id AS calc_id, p.name AS producto, b.name AS negocio,
           cr.total_fixed, cr.variable_unit, cr.unit_cost,
           cr.break_even_units, cr.break_even_sales,
           CASE WHEN pi.manual_price IS NOT NULL THEN 'manual' ELSE 'margen' END AS modo_precio
    FROM workspace.innovalab.computed_results cr
    JOIN workspace.innovalab.calculations c ON c.id = cr.calculation_id
    JOIN workspace.innovalab.products p ON p.id = c.product_id
    JOIN workspace.innovalab.businesses b ON b.id = c.business_id
    JOIN workspace.innovalab.pricing_inputs pi ON pi.calculation_id = c.id
    ORDER BY c.id
    LIMIT 20
"""))

In [0]:
# ============================================================
# GRÁFICO: Punto de equilibrio del cálculo X
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

target_calc = "calc_prod_001_A"  # Pan de mesa - precio manual

row = spark.sql(f"""
    SELECT cr.total_fixed, cr.variable_unit, cr.break_even_units, cr.break_even_sales,
           cr.unit_cost, cr.contribution_margin_unit,
           p.name AS product_name, cs.estimated_volume,
           pi.manual_price, pi.expected_margin_pct
    FROM workspace.innovalab.computed_results cr
    JOIN workspace.innovalab.calculations c ON c.id = cr.calculation_id
    JOIN workspace.innovalab.products p ON p.id = c.product_id
    JOIN workspace.innovalab.costing_setup cs ON cs.calculation_id = c.id
    JOIN workspace.innovalab.pricing_inputs pi ON pi.calculation_id = c.id
    WHERE cr.calculation_id = '{target_calc}'
""").collect()[0]

total_fixed = float(row["total_fixed"])
variable_unit = float(row["variable_unit"])
if row["manual_price"]:
    price = float(row["manual_price"])
else:
    price = float(row["unit_cost"]) * (1 + float(row["expected_margin_pct"]))
be_units = float(row["break_even_units"])
be_sales = float(row["break_even_sales"])
volume = float(row["estimated_volume"])
pname = row["product_name"]

x = np.linspace(0, volume * 1.5, 300)
revenue = price * x
total_cost = total_fixed + variable_unit * x

fig, ax = plt.subplots(figsize=(11, 7))
ax.plot(x, revenue, "g-", linewidth=2.5, label=f"Ingresos (${price:,.0f}/unidad)")
ax.plot(x, total_cost, "r-", linewidth=2.5, label=f"Costo total (CV ${variable_unit:,.0f}/u + CF ${total_fixed:,.0f})")
ax.axhline(y=total_fixed, color="orange", linestyle="--", alpha=0.6, label=f"Costo fijo = ${total_fixed:,.0f}")
ax.plot(be_units, be_sales, "ko", markersize=12, zorder=5)
ax.annotate(f"Punto de equilibrio\n{be_units:,.0f} unidades\n${be_sales:,.0f}",
            xy=(be_units, be_sales),
            xytext=(be_units + volume * 0.15, be_sales - total_fixed * 0.4),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
            fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.8))
ax.fill_between(x, revenue, total_cost, where=(revenue >= total_cost), alpha=0.1, color="green", label="Ganancia")
ax.fill_between(x, revenue, total_cost, where=(revenue < total_cost), alpha=0.1, color="red", label="Pérdida")
ax.set_xlabel("Unidades vendidas", fontsize=12)
ax.set_ylabel("$ (ARS)", fontsize=12)
ax.set_title(f"Punto de Equilibrio — {pname} ({target_calc})", fontsize=14)
ax.legend(loc="upper left", fontsize=10)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
plt.tight_layout()
plt.show()

## Análisis y Sugerencias

### Validación de la lógica

✅ **Orden de inserción correcto**: profiles → businesses → products → business_cost_lines → calculations → costing_setup/scenarios → calc_cost_lines → pricing_inputs → computed_results. Todos los padres se insertan antes que sus hijos.

✅ **Cálculo del punto de equilibrio**:
- `total_fixed` se obtiene sumando las líneas de costo con `behavior='fixed'` (ya prorrateadas con `allocation_pct`)
- `variable_unit` se obtiene sumando las líneas con `behavior='variable'` (costo por unidad)
- `price` usa `manual_price` si existe; si no, `unit_cost × (1 + expected_margin_pct)`
- `break_even_units = total_fixed / (price - variable_unit)` ✓
- `break_even_sales = break_even_units × price` ✓

### Observaciones y sugerencias de mejora

1. **`business_cost_lines.amount_period` para costos variables**: Actualmente guarda el precio por unidad del insumo. El nombre sugiere "monto por período". Se sugiere agregar un campo `unit_of_measure` o renombrar a `amount` para mayor claridad.

2. **Prorrateo de costos fijos (`allocation_pct`)**: Se usa un porcentaje fijo por producto. Sería más preciso permitir prorrateos basados en volumen de producción (unidades producidas / volumen total del negocio).

3. **`computed_results` no persiste el `price`**: El precio calculado no se guarda como columna. Se puede derivar de `break_even_sales / break_even_units`, pero sería más explícito agregar una columna `price`.

4. **Escenarios múltiples**: El modelo soporta múltiples escenarios por cálculo, pero solo se creó el escenario base. Sería valioso agregar escenarios optimista y pesimista para análisis de sensibilidad.

5. **Margen de contribución negativo**: Si `price < variable_unit`, el punto de equilibrio no existe (pérdida en cada unidad). El modelo actual devuelve 0 en `break_even_units`, lo cual puede ser engañoso. Se sugiere devolver `NULL` o un valor negativo con un flag de alerta.

6. **`traceability` (direct/indirect)**: Está presente pero no se usa en el cálculo. Se podría usar para asignar costos indirectos mediante bases de distribución (ej. horas máquina, m² de superficie).

7. **`costing_setup.estimated_volume`**: Es clave para calcular el costo unitario. Se sugiere validar que sea > 0 con un constraint o CHECK.

8. **`results_payload`**: Se podría usar para guardar un JSON con el desglose detallado (cada línea de costo con su monto) para trazabilidad auditiva del cálculo.

9. **`pricing_inputs.margin_convention`**: Actualmente permite 'manual' y 'markup_on_cost'. Se sugiere agregar 'margin_on_price' (margen sobre precio) ya que es una convención común que difiere del markup sobre costo.

In [0]:
# ============================================================
# GRÁFICO COMPARATIVO: _A (precio manual) vs _B (margen sobre costo)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

target_product = "prod_001"  # Pan de mesa

rows = spark.sql(f"""
    SELECT cr.calculation_id, cr.total_fixed, cr.variable_unit,
           cr.break_even_units, cr.break_even_sales,
           cr.unit_cost, cs.estimated_volume,
           pi.manual_price, pi.expected_margin_pct,
           p.name AS product_name
    FROM workspace.innovalab.computed_results cr
    JOIN workspace.innovalab.calculations c ON c.id = cr.calculation_id
    JOIN workspace.innovalab.products p ON p.id = c.product_id
    JOIN workspace.innovalab.costing_setup cs ON cs.calculation_id = c.id
    JOIN workspace.innovalab.pricing_inputs pi ON pi.calculation_id = c.id
    WHERE c.product_id = '{target_product}'
    ORDER BY cr.calculation_id
""").collect()

fig, ax = plt.subplots(figsize=(12, 7))
colors = {"A": ("#2ca02c", "#1a6b1a"), "B": ("#1f77b4", "#0d4d8a")}
fixed_cost = float(rows[0]["total_fixed"])
pname = rows[0]["product_name"]

for row in rows:
    suffix = "A" if row["calculation_id"].endswith("_A") else "B"
    tf = float(row["total_fixed"])
    vu = float(row["variable_unit"])
    vol = float(row["estimated_volume"])

    if row["manual_price"]:
        price = float(row["manual_price"])
        price_label = f"${price:,.0f} (manual)"
    else:
        price = float(row["unit_cost"]) * (1 + float(row["expected_margin_pct"]))
        price_label = f"${price:,.0f} (margen {float(row['expected_margin_pct'])*100:.0f}%)"

    be_u = float(row["break_even_units"])
    be_s = float(row["break_even_sales"])
    mode = "Precio manual" if suffix == "A" else "Margen s/costo"
    c_rev, c_cost = colors[suffix]

    x = np.linspace(0, vol * 1.5, 300)
    ax.plot(x, price * x, "-", color=c_rev, linewidth=2.2,
            label=f"Ingresos {mode} ({price_label})")
    ax.plot(x, tf + vu * x, "--", color=c_cost, linewidth=2.2,
            label=f"Costo total {mode} (CV ${vu:,.0f}/u)")
    ax.plot(be_u, be_s, "o", color=c_cost, markersize=10, zorder=5)
    y_offset = be_s * 0.55 if suffix == "B" else be_s * 1.12
    ax.annotate(f"BE {mode}\n{be_u:,.0f} u\n${be_s:,.0f}",
                xy=(be_u, be_s),
                xytext=(be_u + vol * 0.08, y_offset),
                arrowprops=dict(arrowstyle="->", color=c_cost, lw=1.5),
                fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.85))

ax.axhline(y=fixed_cost, color="orange", linestyle=":", alpha=0.5, label=f"Costo fijo = ${fixed_cost:,.0f}")
ax.set_xlabel("Unidades vendidas", fontsize=12)
ax.set_ylabel("$ (ARS)", fontsize=12)
ax.set_title(f"Comparativa A vs B — {pname}", fontsize=14)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# BREAK-EVEN: Pan Artesano (datos del documento "Analisis basico")
# ============================================================

# Diagnóstico: verificar que Spark ve los datos de Pan Artesano
diag = {
    "calc_cost_lines": "calculation_id",
    "pricing_inputs": "calculation_id",
    "scenarios": "calculation_id",
    "costing_setup": "calculation_id",
    "calculations": "id"
}
for t, col in diag.items():
    n = spark.sql(f"SELECT COUNT(*) AS n FROM workspace.innovalab.{t} WHERE {col} LIKE 'calc_prod_02%'").collect()[0]["n"]
    print(f"  {t}: {n} rows for Pan Artesano")

# Eliminar computed_results existentes para Pan Artesano (evitar duplicados)
spark.sql("DELETE FROM workspace.innovalab.computed_results WHERE calculation_id LIKE 'calc_prod_02%'")

# Calcular e insertar resultados
spark.sql("""
INSERT INTO workspace.innovalab.computed_results
SELECT
    uuid() AS id,
    t.calculation_id,
    t.scenario_id,
    CAST(t.total_fixed AS DECIMAL(18,4)),
    CAST(t.variable_unit AS DECIMAL(18,4)),
    CAST(t.total_fixed + t.variable_unit * t.estimated_volume AS DECIMAL(18,4)) AS total_cost,
    CAST(t.total_fixed / t.estimated_volume + t.variable_unit AS DECIMAL(18,4)) AS unit_cost,
    CAST(t.price - t.variable_unit AS DECIMAL(18,4)) AS contribution_margin_unit,
    CASE WHEN t.price - t.variable_unit > 0
         THEN CAST(t.total_fixed / (t.price - t.variable_unit) AS DECIMAL(18,4))
         ELSE CAST(0 AS DECIMAL(18,4)) END AS break_even_units,
    CASE WHEN t.price - t.variable_unit > 0
         THEN CAST(t.total_fixed / (t.price - t.variable_unit) * t.price AS DECIMAL(18,4))
         ELSE CAST(0 AS DECIMAL(18,4)) END AS break_even_sales,
    current_timestamp() AS computed_at,
    NULL AS results_payload
FROM (
    SELECT
        c.id AS calculation_id,
        sc.id AS scenario_id,
        tf.total_fixed,
        vu.variable_unit,
        cs.estimated_volume,
        COALESCE(
            pi.manual_price,
            (tf.total_fixed / cs.estimated_volume + vu.variable_unit) * (1 + pi.expected_margin_pct)
        ) AS price
    FROM workspace.innovalab.calculations c
    JOIN workspace.innovalab.scenarios sc ON sc.calculation_id = c.id AND sc.is_base = true
    JOIN workspace.innovalab.costing_setup cs ON cs.calculation_id = c.id
    JOIN workspace.innovalab.pricing_inputs pi ON pi.calculation_id = c.id
    JOIN (
        SELECT calculation_id, scenario_id, SUM(amount) AS total_fixed
        FROM workspace.innovalab.calc_cost_lines
        WHERE behavior = 'fixed'
        GROUP BY calculation_id, scenario_id
    ) tf ON tf.calculation_id = c.id AND tf.scenario_id = sc.id
    JOIN (
        SELECT calculation_id, scenario_id, SUM(amount) AS variable_unit
        FROM workspace.innovalab.calc_cost_lines
        WHERE behavior = 'variable'
        GROUP BY calculation_id, scenario_id
    ) vu ON vu.calculation_id = c.id AND vu.scenario_id = sc.id
    WHERE c.id LIKE 'calc_prod_02%'
) t
""")

# Mostrar resultados
print("=" * 80)
print("PUNTO DE EQUILIBRIO - Pan Artesano (4 productos x 2 escenarios)")
print("=" * 80)
display(spark.sql("""
SELECT
    p.name AS producto,
    cr.calculation_id,
    CASE WHEN pi.margin_convention = 'manual'
         THEN CONCAT('$', CAST(pi.manual_price AS INT))
         ELSE CONCAT('markup ', CAST(pi.expected_margin_pct * 100 AS INT), '%')
    END AS esquema_precio,
    CAST(cr.total_fixed AS BIGINT) AS costos_fijos,
    CAST(cr.variable_unit AS DECIMAL(18,2)) AS costo_var_unit,
    CAST(cr.unit_cost AS DECIMAL(18,2)) AS costo_completo,
    CAST(cr.contribution_margin_unit AS DECIMAL(18,2)) AS margen_contrib,
    CAST(cr.break_even_units AS INT) AS eq_unidades,
    CAST(cr.break_even_sales AS BIGINT) AS eq_facturacion,
    cs.estimated_volume AS vol_estimada
FROM workspace.innovalab.computed_results cr
JOIN workspace.innovalab.calculations c ON c.id = cr.calculation_id
JOIN workspace.innovalab.products p ON p.id = c.product_id
JOIN workspace.innovalab.costing_setup cs ON cs.calculation_id = c.id
JOIN workspace.innovalab.pricing_inputs pi ON pi.calculation_id = c.id
WHERE cr.calculation_id LIKE 'calc_prod_02%'
ORDER BY p.name, cr.calculation_id
"""))

In [0]:
# ============================================================
# GRÁFICO COMPARATIVO: _A (precio manual) vs _B (margen sobre costo)
# Pan Lactal de Avena — datos del documento "Analisis basico"
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

target_product = "prod_021"  # Pan lactal de avena

rows = spark.sql(f"""
    SELECT cr.calculation_id, cr.total_fixed, cr.variable_unit,
           cr.break_even_units, cr.break_even_sales,
           cr.unit_cost, cs.estimated_volume,
           pi.manual_price, pi.expected_margin_pct,
           p.name AS product_name
    FROM workspace.innovalab.computed_results cr
    JOIN workspace.innovalab.calculations c ON c.id = cr.calculation_id
    JOIN workspace.innovalab.products p ON p.id = c.product_id
    JOIN workspace.innovalab.costing_setup cs ON cs.calculation_id = c.id
    JOIN workspace.innovalab.pricing_inputs pi ON pi.calculation_id = c.id
    WHERE c.product_id = '{target_product}'
    ORDER BY cr.calculation_id
""").collect()

fig, ax = plt.subplots(figsize=(12, 7))
colors = {"A": ("#2ca02c", "#1a6b1a"), "B": ("#1f77b4", "#0d4d8a")}
fixed_cost = float(rows[0]["total_fixed"])
pname = rows[0]["product_name"]

for row in rows:
    suffix = "A" if row["calculation_id"].endswith("_A") else "B"
    tf = float(row["total_fixed"])
    vu = float(row["variable_unit"])
    vol = float(row["estimated_volume"])

    if row["manual_price"]:
        price = float(row["manual_price"])
        price_label = f"${price:,.0f} (manual)"
    else:
        price = float(row["unit_cost"]) * (1 + float(row["expected_margin_pct"]))
        price_label = f"${price:,.0f} (margen {float(row['expected_margin_pct'])*100:.0f}%)"

    be_u = float(row["break_even_units"])
    be_s = float(row["break_even_sales"])
    mode = "Precio manual" if suffix == "A" else "Margen s/costo"
    c_rev, c_cost = colors[suffix]

    x = np.linspace(0, max(be_u * 2, vol * 1.5), 300)
    ax.plot(x, price * x, "-", color=c_rev, linewidth=2.2,
            label=f"Ingresos {mode} ({price_label})")
    ax.plot(x, tf + vu * x, "--", color=c_cost, linewidth=2.2,
            label=f"Costo total {mode} (CV ${vu:,.0f}/u)")
    ax.plot(be_u, be_s, "o", color=c_cost, markersize=10, zorder=5)
    y_offset = be_s * 0.55 if suffix == "B" else be_s * 1.12
    ax.annotate(f"BE {mode}\n{be_u:,.0f} u\n${be_s:,.0f}",
                xy=(be_u, be_s),
                xytext=(be_u + vol * 0.08, y_offset),
                arrowprops=dict(arrowstyle="->", color=c_cost, lw=1.5),
                fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.85))

ax.axhline(y=fixed_cost, color="orange", linestyle=":", alpha=0.5, label=f"Costo fijo = ${fixed_cost:,.0f}")
ax.set_xlabel("Unidades vendidas", fontsize=12)
ax.set_ylabel("$ (ARS)", fontsize=12)
ax.set_title(f"Comparativa A vs B — {pname}", fontsize=14)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# GRÁFICO COMPARATIVO: _A (precio manual) vs _B (margen sobre costo)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

target_product = "prod_014"  # Silla de Madera

rows = spark.sql(f"""
    SELECT cr.calculation_id, cr.total_fixed, cr.variable_unit,
           cr.break_even_units, cr.break_even_sales,
           cr.unit_cost, cs.estimated_volume,
           pi.manual_price, pi.expected_margin_pct,
           p.name AS product_name
    FROM workspace.innovalab.computed_results cr
    JOIN workspace.innovalab.calculations c ON c.id = cr.calculation_id
    JOIN workspace.innovalab.products p ON p.id = c.product_id
    JOIN workspace.innovalab.costing_setup cs ON cs.calculation_id = c.id
    JOIN workspace.innovalab.pricing_inputs pi ON pi.calculation_id = c.id
    WHERE c.product_id = '{target_product}'
    ORDER BY cr.calculation_id
""").collect()

fig, ax = plt.subplots(figsize=(12, 7))
colors = {"A": ("#2ca02c", "#1a6b1a"), "B": ("#1f77b4", "#0d4d8a")}
fixed_cost = float(rows[0]["total_fixed"])
pname = rows[0]["product_name"]

for row in rows:
    suffix = "A" if row["calculation_id"].endswith("_A") else "B"
    tf = float(row["total_fixed"])
    vu = float(row["variable_unit"])
    vol = float(row["estimated_volume"])

    if row["manual_price"]:
        price = float(row["manual_price"])
        price_label = f"${price:,.0f} (manual)"
    else:
        price = float(row["unit_cost"]) * (1 + float(row["expected_margin_pct"]))
        price_label = f"${price:,.0f} (margen {float(row['expected_margin_pct'])*100:.0f}%)"

    be_u = float(row["break_even_units"])
    be_s = float(row["break_even_sales"])
    mode = "Precio manual" if suffix == "A" else "Margen s/costo"
    c_rev, c_cost = colors[suffix]

    x = np.linspace(0, vol * 1.5, 300)
    ax.plot(x, price * x, "-", color=c_rev, linewidth=2.2,
            label=f"Ingresos {mode} ({price_label})")
    ax.plot(x, tf + vu * x, "--", color=c_cost, linewidth=2.2,
            label=f"Costo total {mode} (CV ${vu:,.0f}/u)")
    ax.plot(be_u, be_s, "o", color=c_cost, markersize=10, zorder=5)
    y_offset = be_s * 0.55 if suffix == "B" else be_s * 1.12
    ax.annotate(f"BE {mode}\n{be_u:,.0f} u\n${be_s:,.0f}",
                xy=(be_u, be_s),
                xytext=(be_u + vol * 0.08, y_offset),
                arrowprops=dict(arrowstyle="->", color=c_cost, lw=1.5),
                fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.85))

ax.axhline(y=fixed_cost, color="orange", linestyle=":", alpha=0.5, label=f"Costo fijo = ${fixed_cost:,.0f}")
ax.set_xlabel("Unidades vendidas", fontsize=12)
ax.set_ylabel("$ (ARS)", fontsize=12)
ax.set_title(f"Comparativa A vs B — {pname}", fontsize=14)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
plt.tight_layout()
plt.show()